# Agentic RAG — Let the LLM Pick the Best Technique

**Problem:** With 10 retrieval techniques available, how do you know which one to use?

**Solution:** Let an LLM agent analyze the query and automatically select the best technique.

The agent acts as a **router** — it reads the query, understands what kind of
information need it represents, and dispatches to the right retriever.

**Pipeline:** Query → LLM Agent (picks technique) → Run chosen technique → Results

In [ ]:
# The agent's decision framework
technique_rules = {
    "bm25": {
        "when": "Exact keywords, specific terms, names, numbers, codes",
        "examples": ["metformin 500mg", "reliability below 90%", "supplier ID SC-004"],
    },
    "semantic": {
        "when": "Conceptual queries, synonyms, meaning-based matching",
        "examples": ["Find vendors in China", "treatment for high blood sugar"],
    },
    "hybrid": {
        "when": "Need both exact keyword AND meaning matching",
        "examples": ["TSMC semiconductor reliability", "warfarin drug interactions"],
    },
    "query_rewriter": {
        "when": "Vague, incomplete, or poorly worded queries",
        "examples": ["revolution farming", "medicine for sugar"],
    },
    "multi_query": {
        "when": "Multiple aspects, constraints, or comparison queries",
        "examples": ["blood thinners AND painkillers", "tall buildings in NYC in the 1930s"],
    },
    "hyde": {
        "when": "Question-style query searching statement-style documents",
        "examples": ["Explain photosynthesis", "What is CRISPR?"],
    },
    "graph_rag": {
        "when": "Multi-hop reasoning, entity relationships, cause-effect chains",
        "examples": ["What happens if TSMC shuts down?", "Which companies depend on Bosch?"],
    },
}

print("Agent's technique selection guide:\n")
for tech, info in technique_rules.items():
    print(f"  {tech}:")
    print(f"    When: {info['when']}")
    print(f"    Examples: {info['examples']}")
    print()

In [ ]:
# Simulating agent decisions (without actual LLM)
def simple_agent(query):
    """Rule-based agent that mimics what the LLM would decide."""
    q = query.lower()
    
    # Check for exact terms / numbers
    import re
    if re.search(r'\d+mg|\d+%|below \d+|above \d+', q):
        return "bm25", "Query contains specific numbers/dosages → exact keyword matching"
    
    # Check for question-style (What/How/Explain)
    if q.startswith(("explain", "what is", "how does", "how do")):
        return "hyde", "Question-style query → generate hypothetical answer first"
    
    # Check for multi-hop / cause-effect
    if "what happens if" in q or "depend" in q or "impact" in q:
        return "graph_rag", "Cause-effect / multi-hop query → graph traversal"
    
    # Check for multiple aspects (AND, between, both)
    if " and " in q or "between" in q or "both" in q:
        return "multi_query", "Multiple aspects detected → decompose into sub-queries"
    
    # Check for vague / very short queries
    if len(q.split()) <= 3:
        return "query_rewriter", "Short/vague query → LLM expansion"
    
    # Default
    return "hybrid", "General query → hybrid search for balanced results"


test_queries = [
    "metformin 500mg side effects",
    "Find vendors in China",
    "Explain photosynthesis",
    "What happens if TSMC shuts down?",
    "Drug interactions between blood thinners and painkillers",
    "revolution farming",
    "TSMC semiconductor reliability",
]

print("Agent routing decisions:\n")
for query in test_queries:
    technique, reasoning = simple_agent(query)
    print(f"  Query: \"{query}\"")
    print(f"  → Agent chose: {technique}")
    print(f"  → Reasoning: {reasoning}\n")

## Rule-Based vs LLM Agent

The `simple_agent()` above uses if/else rules. The real implementation uses an LLM:

```
You are a retrieval technique selector. Given a query, pick the BEST technique:
1. bm25 — exact keywords, numbers, names
2. semantic — conceptual, synonyms
3. hybrid — both keyword and meaning
...

Query: "Drug interactions between blood thinners and painkillers"
Best technique:
```

**LLM advantages over rules:**
- Handles ambiguous queries better
- Understands context and nuance
- No need to maintain complex rule sets

**LLM disadvantages:**
- Adds latency (~200-500ms per routing decision)
- Can make wrong choices
- Non-deterministic

In [ ]:
# The full agentic pipeline visualized
print("=" * 60)
print("AGENTIC RAG — Full Pipeline")
print("=" * 60)
print()
print("  User Query")
print("      │")
print("      ▼")
print("  ┌─────────────────┐")
print("  │   LLM Agent     │  ← Analyzes query type")
print("  │   (Router)      │")
print("  └────────┬────────┘")
print("           │")
print("     ┌─────┼─────┬─────────┐")
print("     ▼     ▼     ▼         ▼")
print("  ┌─────┐┌────┐┌──────┐┌───────┐")
print("  │BM25 ││Sem.││Hybrid││Graph  │ ...")
print("  └──┬──┘└─┬──┘└──┬───┘└───┬───┘")
print("     └─────┴──────┴────────┘")
print("           │")
print("           ▼")
print("    Results + Explanation")
print("    (includes WHY agent")
print("     chose this technique)")

## Key Takeaways

1. **Agentic RAG = meta-technique** — it picks the best technique per query
2. **No single technique is best for all queries** — that's the whole point
3. **The agent can be rule-based or LLM-based** — LLM is more flexible but slower
4. **Transparency matters** — the agent should explain WHY it chose a technique
5. **This is the "I don't know which to use" answer** — let the AI decide
6. **Future: agents can use MULTIPLE techniques** — run BM25 + Semantic, then re-rank